In [1]:
#%%
import os
import sys
import bisect
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
import pandas as pd
import numpy as np
import pyBigWig
from pyfaidx import Fasta
from torchmetrics import PearsonCorrCoef
from tqdm import tqdm
from pathlib import Path
from typing import List, Dict, Callable, Any, cast
import functools
from accelerate import Accelerator
from safetensors.torch import load_model, save_model
from accelerate.utils import DistributedDataParallelKwargs
from typing import List


/root/miniconda3/envs/caduceus_env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ==============================================================================
# 🌟 1. Global Configuration and Environment Initialization
# ==============================================================================
torch.set_float32_matmul_precision('high')
ddp_kwargs = DistributedDataParallelKwargs(find_unused_parameters=True)
accelerator = Accelerator(mixed_precision="bf16", kwargs_handlers=[ddp_kwargs])

device = accelerator.device
accelerator.print(f"🚀 Accelerate initialized. Current device: {device}, Available GPUs: {torch.cuda.device_count()}")

config = {
    "model_name": "PlantGenoANN", 
    "best_model_checkpoint_path": "./ath",
    
    "resume_from_checkpoint": True,
    "checkpoint_dir": "/root/autodl-tmp/test_ath_github/test_ath",        
    
    "species": "arabidopsis",
    
    "sequence_length": 32_768,
    "keep_target_center_fraction": 0.375,
    "train_overlap": 0.999, 
    

    "mini_batch_size": 2,             
    "num_accumulation_gradient": 16,   
    "num_steps_training": 2022,      
    "log_every_n_steps": 50,
    

    "weight_decay": 0.01,
    "initial_learning_rate": 1e-5,
    "num_steps_warmup": 50,          
    "end_learning_rate": 5e-5,        
    
    "validate_every_n_steps": 50, 
    
    "seed": 0,
    "num_workers": 8,                
}

torch.manual_seed(config["seed"])
np.random.seed(config["seed"])


🚀 Accelerate initialized. Current device: cuda, Available GPUs: 1


In [3]:
# ==============================================================================
# 🌟 2. Localized Hard-bound Imports (Specific to Pre-trained Model)
# ==============================================================================
from transformers import AutoModel, AutoTokenizer, AutoConfig

# Local data paths
FASTA_PATH = './ath/genome.fasta'
BW_DIR = './ath'
SPLITS_PATH = './ath/splits.bed'
METADATA_PATH = './ath/benchmark_metadata.tsv'

bigwig_ids = ['SRR13808079','SRR13808080','SRR13808081','SRR13808084']
actual_bigwig_paths = [os.path.join(BW_DIR, f"{bw_id}.bigwig") for bw_id in bigwig_ids]


In [4]:
# ==============================================================================
# 🌟 3. Model Architecture Definition (PreTrained without Species ID)
# ==============================================================================
def crop_center(x: torch.Tensor, keep_target_center_fraction: float) -> torch.Tensor:
    seq_len = x.shape[-2]
    target_offset = int(seq_len * (1 - keep_target_center_fraction) // 2)
    target_length = seq_len - 2 * target_offset
    return x[..., target_offset:target_offset + target_length, :]

class LinearHead(nn.Module):
    def __init__(self, embed_dim: int, num_labels: int):
        super().__init__()
        self.layer_norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_labels)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.softplus(self.head(self.layer_norm(x)))

class HFModelWithHead(nn.Module):
    def __init__(self, model_name: str, bigwig_track_names: List[str], keep_target_center_fraction: float):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name, trust_remote_code=True, local_files_only=True)
        self.backbone = AutoModel.from_pretrained(model_name, trust_remote_code=True, local_files_only=True)
        self.keep_target_center_fraction = keep_target_center_fraction
        self.bigwig_head = LinearHead(2 * self.config.d_model, len(bigwig_track_names))
    
    def forward(self, tokens: torch.Tensor, **kwargs) -> Dict[str, torch.Tensor]:
        outputs = self.backbone(input_ids=tokens)
        embedding = outputs["hidden_states"]
        if self.keep_target_center_fraction < 1.0:
            embedding = crop_center(embedding, self.keep_target_center_fraction)
        return {"bigwig_tracks_logits": self.bigwig_head(embedding)}

tokenizer = AutoTokenizer.from_pretrained(config["model_name"],trust_remote_code=True, local_files_only=True)
model = HFModelWithHead(config["model_name"], bigwig_ids, config["keep_target_center_fraction"])


In [5]:
# ==============================================================================
# 🌟 4. Dataset and DataLoader
# ==============================================================================
_fasta_cache, _bigwig_cache = {}, {}

def _get_fasta_handle(fasta_path: str) -> Fasta:
    key = (os.getpid(), str(Path(fasta_path).resolve()))
    if key not in _fasta_cache: _fasta_cache[key] = Fasta(key[1], as_raw=True, sequence_always_upper=True)
    return _fasta_cache[key]

def _get_bigwig_handle(bigwig_path: str) -> pyBigWig.pyBigWig:
    key = (os.getpid(), str(Path(bigwig_path).resolve()))
    if key not in _bigwig_cache: _bigwig_cache[key] = pyBigWig.open(key[1])
    return _bigwig_cache[key]

class GenomeBigWigDataset(Dataset):
    def __init__(self, fasta_path: str, bigwig_path_list, chrom_regions: pd.DataFrame, 
                 split: str, sequence_length: int, tokenizer, transform_fn: Callable, 
                 overlap: float = 0.0, keep_target_center_fraction: float = 1.0):
        super().__init__()
        self.fasta_path, self.bigwig_path_list = fasta_path, bigwig_path_list
        self.sequence_length, self.tokenizer = sequence_length, tokenizer
        self.transform_fn, self.keep_target_center_fraction = transform_fn, keep_target_center_fraction
        self.stride = max(1, int((1 - overlap) * sequence_length))
        
        split_regions = chrom_regions[chrom_regions["split"] == split].copy()
        region_list = [(row.chr_name, row.start, row.end) for _, row in split_regions.iterrows()]
            
        self.chromosome_info, self._cumulative_starts, self.num_samples = self._process_regions(region_list)

    def _process_regions(self, actual_regions_list) -> tuple:
        region_info, cumulative_starts, total_sequences = [], [], 0
        for chr_name, region_s, region_e in actual_regions_list:
            num_sequences = max(0, (region_e - region_s - self.sequence_length) // self.stride + 1)
            if num_sequences > 0:
                region_info.append({"chr_name": chr_name, "region_start_offset": region_s})
                cumulative_starts.append(total_sequences)
                total_sequences += num_sequences
        return region_info, cumulative_starts, int(total_sequences)

    def __len__(self): return self.num_samples

    def __getitem__(self, idx):
        chromosome_idx = bisect.bisect_right(self._cumulative_starts, idx) - 1
        chrom = self.chromosome_info[chromosome_idx]["chr_name"]
        start = self.chromosome_info[chromosome_idx]["region_start_offset"] + (idx - self._cumulative_starts[chromosome_idx]) * self.stride
        end = start + self.sequence_length

        fasta = _get_fasta_handle(self.fasta_path)
        tokens = self.tokenizer(fasta[chrom][start:end], padding="max_length", truncation=True, max_length=self.sequence_length+2, return_tensors="pt")["input_ids"][0]

        targets = np.array([_get_bigwig_handle(bw).values(chrom, start, end, numpy=True) for bw in self.bigwig_path_list]).T
        targets = torch.nan_to_num(torch.tensor(targets, dtype=torch.float32), nan=0.0)
        
        if self.keep_target_center_fraction < 1.0: 
            targets = crop_center(targets, self.keep_target_center_fraction)
        return {"tokens": tokens, "bigwig_targets": self.transform_fn(targets)}

def create_targets_scaling_fn(metadata_df: pd.DataFrame):
    means = torch.tensor(metadata_df["mean"].to_numpy(), dtype=torch.float32)
    def transform_fn(x: torch.Tensor) -> torch.Tensor:
        scaled = x / means.to(x.device)
        return torch.where(scaled > 10.0, 2.0 * torch.sqrt(scaled * 10.0) - 10.0, scaled)
    return transform_fn

species_splits_df = pd.read_csv(SPLITS_PATH, sep="\t", header=None, names=["chr_name", "start", "end", "split"], dtype={"chr_name": str, "start": int, "end": int, "split": str})
metadata_df = pd.read_csv(METADATA_PATH, sep="\t")
metadata_df = metadata_df[metadata_df["species_common_name"] == config["species"]].set_index("file_id").loc[bigwig_ids].reset_index()

create_dataset_fn = functools.partial(
    GenomeBigWigDataset, fasta_path=FASTA_PATH, bigwig_path_list=actual_bigwig_paths, 
    chrom_regions=species_splits_df, sequence_length=config["sequence_length"], 
    tokenizer=tokenizer, transform_fn=create_targets_scaling_fn(metadata_df), 
    keep_target_center_fraction=config["keep_target_center_fraction"]
)

train_dataset = create_dataset_fn(split="train", overlap=config["train_overlap"])
val_dataset = create_dataset_fn(split="val")
test_dataset = create_dataset_fn(split="test")

train_loader = DataLoader(train_dataset, batch_size=config["mini_batch_size"], shuffle=True, num_workers=config["num_workers"], pin_memory=True, prefetch_factor=2)
val_loader = DataLoader(val_dataset, batch_size=config["mini_batch_size"], shuffle=False, num_workers=config["num_workers"], pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=config["mini_batch_size"], shuffle=False, num_workers=config["num_workers"], pin_memory=True)

total_params = sum(p.numel() for p in model.parameters())
accelerator.print("\n" + "="*50)
accelerator.print(f"📊 Model Parameter Statistics (Full Parameter Fine-tuning Mode)")
accelerator.print(f"  - Total parameters: {total_params:,} (100% trainable)")
accelerator.print("="*50 + "\n")

accelerator.print("\n" + "="*50 + "\n📦 Dataset Loaded Successfully\n" + "="*50)
accelerator.print(f"  - Train dataset samples: {len(train_dataset):,}")
accelerator.print(f"  - Validation dataset samples: {len(val_dataset):,}")
accelerator.print(f"  - Test dataset samples: {len(test_dataset):,}")



📊 Model Parameter Statistics (Full Parameter Fine-tuning Mode)
  - Total parameters: 184,884,238 (100% trainable)


📦 Dataset Loaded Successfully
  - Train dataset samples: 3,305,818
  - Validation dataset samples: 181
  - Test dataset samples: 211


In [6]:
# ==============================================================================
# 🌟 5. Loss Functions and Evaluation Metrics
# ==============================================================================
def poisson_loss(ytrue: torch.Tensor, ypred: torch.Tensor, epsilon: float = 1e-7) -> torch.Tensor:
    return ypred - ytrue * torch.log(ypred + epsilon)

def safe_for_grad_log_torch(x: torch.Tensor) -> torch.Tensor:
    return torch.log(torch.where(x > 0.0, x, torch.ones_like(x)))

def poisson_multinomial_loss(logits: torch.Tensor, targets: torch.Tensor, shape_loss_coefficient: float = 5.0, epsilon: float = 1e-7) -> torch.Tensor:
    batch_size, seq_length, num_tracks = logits.shape
    sum_pred, sum_true = logits.sum(dim=1), targets.sum(dim=1) 
    scale_loss = (poisson_loss(sum_true, sum_pred, epsilon=epsilon) / (seq_length + epsilon)).mean()
    
    predicted_counts = logits + epsilon
    targets_with_epsilon = targets + epsilon
    denom = predicted_counts.sum(dim=1, keepdim=True) + epsilon 
    p_pred = predicted_counts / denom
    shape_loss = -(targets_with_epsilon * safe_for_grad_log_torch(p_pred)).sum() / (batch_size * seq_length * num_tracks + epsilon)
    return shape_loss + scale_loss / shape_loss_coefficient

class TracksMetrics:
    def __init__(self, track_names, split: str):
        self.track_names, self.num_tracks, self.split = track_names, len(track_names), split
        self.pearson = PearsonCorrCoef(num_outputs=self.num_tracks).to(device).set_dtype(torch.float64)
        self.losses, self.step_idxs, self.mean_pearsons, self.mean_losses = [], [], [], []
    
    def reset(self): self.pearson.reset(); self.losses = []
    
    def update(self, predictions: torch.Tensor, targets: torch.Tensor, loss: float):
        self.pearson.update(predictions.detach().reshape(-1, self.num_tracks).to(torch.float64), targets.detach().reshape(-1, self.num_tracks).to(torch.float64))
        self.losses.append(loss)
    
    def compute(self):
        correlations = self.pearson.compute().cpu().numpy()
        local_loss = torch.tensor(np.mean(self.losses) if len(self.losses)>0 else 0.0, device=device)
        global_loss = accelerator.reduce(local_loss, reduction="mean")
        res = {f"{n}/pearson": correlations[i] for i, n in enumerate(self.track_names)}
        res["mean/pearson"], res["loss"] = correlations.mean(), global_loss.item()
        return res

    def update_mean_metrics(self, step_idx: int):
        metrics_dict = self.compute()
        self.step_idxs.append(step_idx)
        self.mean_pearsons.append(metrics_dict["mean/pearson"])
        self.mean_losses.append(metrics_dict["loss"])
        if accelerator.is_main_process:
            pd.DataFrame({"step": self.step_idxs, "mean_loss": self.mean_losses, "mean_pearson": self.mean_pearsons}).to_csv(f"metrics_{self.split}.csv", index=False)
        return metrics_dict
        
    def print_metrics(self, metrics_dict, print_per_track, current_lr=None):
        lr_str = f"LR: {current_lr:.2e} | " if current_lr is not None else ""
        if accelerator.is_main_process:
            tqdm.write(f"\n[Step {self.step_idxs[-1]}/{config['num_steps_training']}] {lr_str}Loss: {self.mean_losses[-1]:.4f} | Mean PCC: {self.mean_pearsons[-1]:.4f}")
            if print_per_track:
                for k, v in metrics_dict.items(): 
                    if k not in ["mean/pearson", "loss"]: tqdm.write(f"    {k}: {v:.4f}")

train_metrics, val_metrics, test_metrics = TracksMetrics(bigwig_ids, "train"), TracksMetrics(bigwig_ids, "val"), TracksMetrics(bigwig_ids, "test")


In [7]:
# ==============================================================================
# 🌟 6. Optimizer, Learning Rate, and Accelerate Preparation
# ==============================================================================
optimizer_lr = config["end_learning_rate"]
optimizer = AdamW(model.parameters(), lr=optimizer_lr, weight_decay=config["weight_decay"])

alpha_polynomial_decay = np.log(1.0 / 0.5) / np.log(float(config["num_steps_training"]) / float(config["num_steps_warmup"]))

def _modified_square_decay(current_step: int) -> float:
    if current_step < config["num_steps_warmup"]:
        start_multiplier = config["initial_learning_rate"] / optimizer_lr
        return start_multiplier + (1.0 - start_multiplier) * (float(current_step) / float(config["num_steps_warmup"]))
    decay_multiplier = (float(config["num_steps_warmup"]) / float(current_step + 1)) ** alpha_polynomial_decay
    return min(decay_multiplier, 1.0)

scheduler = LambdaLR(optimizer, lr_lambda=_modified_square_decay)

accelerator.gradient_accumulation_steps = config["num_accumulation_gradient"]
model, optimizer, train_loader, val_loader, test_loader, scheduler = accelerator.prepare(
    model, optimizer, train_loader, val_loader, test_loader, scheduler
)


In [8]:
# ==============================================================================
# 🌟 6.5 Checkpoint Resuming State Management 
# ==============================================================================
class TrainingStateTracker:
    def __init__(self):
        self.starting_step = 0
        self.highest_val_pcc = -float('inf') 
        self.best_step = 0 

    def state_dict(self):
        return {
            "starting_step": self.starting_step,
            "highest_val_pcc": self.highest_val_pcc,
            "best_step": self.best_step
        }

    def load_state_dict(self, state_dict):
        self.starting_step = state_dict["starting_step"]
        self.highest_val_pcc = state_dict.get("highest_val_pcc", -float('inf'))
        self.best_step = state_dict.get("best_step", 0)

# Instantiate the object and register it for Accelerate checkpointing
training_state = TrainingStateTracker()
accelerator.register_for_checkpointing(training_state)

# load checkpoint
if config["resume_from_checkpoint"] and os.path.exists(config["checkpoint_dir"]):
    accelerator.print(f"🔄 Checkpoint detected. Resuming best training state from {config['checkpoint_dir']}...")
    accelerator.load_state(config["checkpoint_dir"])
    accelerator.print(f"✅ Resume successful! Reverting to step {training_state.starting_step} to continue training. Current best validation PCC: {training_state.highest_val_pcc:.4f}")


In [9]:
# ==============================================================================
# 🌟 7. Advanced Training Main Loop
# ==============================================================================
def run_step(model, batch, metrics, is_train=True):
    tokens, targets = batch["tokens"], batch["bigwig_targets"]
    if is_train:
        logits = model(tokens=tokens)["bigwig_tracks_logits"]
        loss = poisson_multinomial_loss(logits=logits, targets=targets)
        accelerator.backward(loss)
    else:
        with torch.inference_mode():
            logits = model(tokens=tokens)["bigwig_tracks_logits"]
            loss = poisson_multinomial_loss(logits=logits, targets=targets)
    metrics.update(predictions=logits.float(), targets=targets.float(), loss=loss.item())

accelerator.print(f"\n🔥 Training started! Total steps: {config['num_steps_training']}, Effective Batch Size: {config['mini_batch_size'] * config['num_accumulation_gradient'] * accelerator.num_processes}")
    
train_iter = iter(train_loader)

model.train()

for step_idx in tqdm(range(training_state.starting_step, config["num_steps_training"]), desc="Training", disable=not accelerator.is_main_process):
    
    training_state.starting_step = step_idx + 1
    
    for _ in range(config["num_accumulation_gradient"]):
        with accelerator.accumulate(model):
            try: batch = next(train_iter)
            except StopIteration: train_iter = iter(train_loader); batch = next(train_iter)
            
            optimizer.zero_grad()
            run_step(model, batch, train_metrics, is_train=True)
            optimizer.step()
            scheduler.step()

    if (step_idx + 1) % config["log_every_n_steps"] == 0:
        metrics_dict = train_metrics.update_mean_metrics(step_idx + 1)
        if accelerator.is_main_process: train_metrics.print_metrics(metrics_dict, print_per_track=False, current_lr=scheduler.get_last_lr()[0])
        train_metrics.reset()
    
    if (step_idx + 1) % config["validate_every_n_steps"] == 0:
        model.eval()
        for val_batch in val_loader: run_step(model, val_batch, val_metrics, is_train=False)
        
        metrics_dict = val_metrics.update_mean_metrics(step_idx + 1)
        current_val_loss = metrics_dict["loss"]
        current_val_pcc = metrics_dict["mean/pearson"]
        
        # 🌟 Update best validation PCC and step
        if current_val_pcc > training_state.highest_val_pcc:
            training_state.highest_val_pcc = current_val_pcc
            training_state.best_step = step_idx + 1
            if accelerator.is_main_process:
                accelerator.print(f"🌟 New best validation performance achieved! PCC: {current_val_pcc:.4f} (Loss: {current_val_loss:.4f})")
                
        if accelerator.is_main_process:
            val_metrics.print_metrics(metrics_dict, print_per_track=True)
            m = accelerator.unwrap_model(model)
            
            # 🌟 Save EVERY checkpoint with a specific step suffix
            current_save_dir = f"{config['best_model_checkpoint_path']}_step_{step_idx + 1}"
            current_ckpt_dir = f"{config['checkpoint_dir']}_step_{step_idx + 1}"
            
            # 1. Save pure model weights
            accelerator.print(f"💾 Saving model checkpoint for step {step_idx + 1} to {current_save_dir}...")
            os.makedirs(current_save_dir, exist_ok=True)
            tokenizer.save_pretrained(current_save_dir)
            m.config.save_pretrained(current_save_dir)
            save_model(m, os.path.join(current_save_dir, "model.safetensors"))
            
            # 2. Save complete training snapshot
            accelerator.print(f"💾 Saving complete training snapshot for step {step_idx + 1}...")
            accelerator.save_state(output_dir=current_ckpt_dir)
                
        val_metrics.reset()
        model.train() 




🔥 Training started! Total steps: 2022, Effective Batch Size: 32


Training:   2%|▏         | 49/2022 [13:43<8:58:15, 16.37s/it]


[Step 50/2022] LR: 4.98e-05 | Loss: 5.0191 | Mean PCC: 0.1735


Training:   2%|▏         | 49/2022 [14:09<8:58:15, 16.37s/it]

🌟 New best validation performance achieved! PCC: 0.3336 (Loss: 0.6044)

[Step 50/2022] Loss: 0.6044 | Mean PCC: 0.3336
    SRR13808079/pearson: 0.3826
    SRR13808080/pearson: 0.3373
    SRR13808081/pearson: 0.2367
    SRR13808084/pearson: 0.3777
💾 Saving model checkpoint for step 50 to /root/autodl-tmp/test_ath_github/test_ath_step_50...


...

Training:  99%|█████████▉| 1999/2022 [9:26:00<06:17, 16.41s/it]


[Step 2000/2022] Loss: 0.5427 | Mean PCC: 0.4351
    SRR13808079/pearson: 0.4860
    SRR13808080/pearson: 0.4325
    SRR13808081/pearson: 0.3081
    SRR13808084/pearson: 0.5137
💾 Saving model checkpoint for step 2000 to /root/autodl-tmp/test_ath_github/test_ath_step_2000...


Removed shared tensor {'backbone.caduceus_ph.backbone.layers.5.mixer.mixer.mamba_rev.in_proj.weight', 'backbone.caduceus_ph.backbone.layers.3.mixer.mixer.mamba_rev.out_proj.weight', 'backbone.caduceus_ph.backbone.layers.13.mixer.mixer.mamba_rev.in_proj.weight', 'backbone.caduceus_ph.backbone.layers.2.mixer.mixer.mamba_rev.out_proj.weight', 'backbone.caduceus_ph.backbone.layers.8.mixer.mixer.mamba_rev.in_proj.weight', 'backbone.caduceus_ph.backbone.layers.15.mixer.mixer.mamba_rev.out_proj.weight', 'backbone.caduceus_ph.backbone.layers.12.mixer.mixer.mamba_rev.in_proj.weight', 'backbone.caduceus_ph.backbone.layers.1.mixer.mixer.mamba_rev.out_proj.weight', 'backbone.caduceus_ph.backbone.layers.4.mixer.mixer.mamba_rev.in_proj.weight', 'backbone.caduceus_ph.backbone.layers.15.mixer.mixer.mamba_rev.in_proj.weight', 'backbone.caduceus_ph.backbone.layers.3.mixer.mixer.mamba_rev.in_proj.weight', 'backbone.caduceus_ph.backbone.layers.5.mixer.mixer.mamba_rev.out_proj.weight', 'backbone.caduceus_p

💾 Saving complete training snapshot for step 2000...


Training: 100%|██████████| 2022/2022 [9:32:04<00:00, 16.98s/it]


In [10]:
# ==============================================================================
# 🌟 8. Final Evaluation on Test Set
# ==============================================================================
accelerator.print("\n" + "="*50 + "\n🧪 Starting final evaluation on the test set\n" + "="*50)
m_eval = accelerator.unwrap_model(model)

# 🌟 Load the best model
if training_state.best_step > 0:
    best_save_dir = f"{config['best_model_checkpoint_path']}_step_{training_state.best_step}"
    safetensors_path = os.path.join(best_save_dir, "model.safetensors")
    accelerator.print(f"📥 Loading best model (PCC: {training_state.highest_val_pcc:.4f}) from step {training_state.best_step} for final test...")
    load_model(m_eval, safetensors_path, strict=True)
else:
    accelerator.print("⚠️ No validation steps or improvements recorded. Using the latest model state for testing.")

model.eval()
for t_batch in tqdm(test_loader, desc="Testing", disable=not accelerator.is_main_process): 
    run_step(model, t_batch, test_metrics, is_train=False)

test_metrics_dict = test_metrics.compute()

if accelerator.is_main_process:
    accelerator.print(f"⭐ Average PCC: {test_metrics_dict['mean/pearson']:.4f} | 📉 Average Loss: {test_metrics_dict['loss']:.4f}\n")
    for track in bigwig_ids: accelerator.print(f"    - {track}: {test_metrics_dict[f'{track}/pearson']:.4f}")
    accelerator.print("\n🎉 All training and evaluation successfully completed!")


🧪 Starting final evaluation on the test set
📥 Loading best model (PCC: 0.4547) from step 1600 for final test...


Testing: 100%|██████████| 106/106 [00:30<00:00,  3.44it/s]

⭐ Average PCC: 0.5257 | 📉 Average Loss: 0.5073

    - SRR13808079: 0.5894
    - SRR13808080: 0.5030
    - SRR13808081: 0.5074
    - SRR13808084: 0.5030

🎉 All training and evaluation successfully completed!
